<a href="https://colab.research.google.com/github/ashishnathani2017/financegpt/blob/master/datacollection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!nvidia-smi | head -5
!pip -q install chromadb sentence-transformers pypdf

Wed Nov 11 18:56:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |


In [3]:
from google.colab import files
import zipfile, pathlib
uploaded = files.upload()  # choose regulatory.zip
zipfile.ZipFile("regulatory.zip").extractall(".")
pdfs = sorted(pathlib.Path("regulatory").rglob("*.pdf"))
print(len(pdfs), "PDFs found")

Saving regulatory.zip to regulatory.zip
16 PDFs found


In [4]:
from pypdf import PdfReader
import re, json

def words(text):
    return re.sub(r"\s+", " ", text).strip().split(" ")

docs, chunks, stats = [], [], []
for p in pdfs:
    text = "\n".join((pg.extract_text() or "") for pg in PdfReader(p).pages)
    w = words(text)
    docs.append({"file": p.name, "chars": len(text), "words": len(w)})
    for i in range(0, len(w), 700):          # ~800 words, 100 overlap
        chunks.append({"doc": p.name, "text": " ".join(w[i:i+800])})
        if i + 800 >= len(w): break
print(f"{len(docs)} documents, {len(chunks)} chunks")
for d in docs: print(d)

16 documents, 123 chunks
{'file': '01MPR071020267B057FA4E71E4164A8B14EDC40197418.pdf', 'chars': 292702, 'words': 44966}
{'file': 'EDITEDTR101020267A1D86E2C75B46B79D8D065509673641.pdf', 'chars': 51079, 'words': 8687}
{'file': 'PR1266EE06BBCD7F5A4B2091FB59ADE87FE4A1.pdf', 'chars': 25114, 'words': 3908}
{'file': 'PR1361B0E4694B08CE42CDBEFE60069A43F7D0.pdf', 'chars': 30728, 'words': 4767}
{'file': 'PR5720E56F0A4611A404EACBD71F2207673A2.pdf', 'chars': 2944, 'words': 365}
{'file': 'PR62462AD650C8168481BAC17613CC54CFED8.pdf', 'chars': 2586, 'words': 348}
{'file': 'PR80907599DE5FD164918A49085C9D6270116.pdf', 'chars': 8863, 'words': 1292}
{'file': 'PR9257E0CB4D769F24658B0AB41428CA46F97.pdf', 'chars': 33836, 'words': 5295}
{'file': 'sebi_Feb_26_2026_Categorization_and_Rationalization_of_Mutual_Fund_Schemes.pdf', 'chars': 45675, 'words': 7137}
{'file': 'sebi_Feb_26_2026_Ease_of_Doing_Investment_EoDI_-_Disclosure_of_registered_name_and_registration_number_by_SEBI_regulated.pdf', 'chars': 7613, 'wo

Build Index

In [5]:
import chromadb
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("BAAI/bge-m3")
client = chromadb.Client()
col = client.create_collection("rbi_sebi", metadata={"hnsw:space": "cosine"})
emb = model.encode([c["text"] for c in chunks], batch_size=8, normalize_embeddings=True).tolist()
col.add(ids=[f"c{i}" for i in range(len(chunks))],
        documents=[c["text"] for c in chunks],
        metadatas=[{"doc": c["doc"]} for c in chunks],
        embeddings=emb)
print("indexed", col.count())


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.27GB            

... (8 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

indexed 123


Test Retrival

In [6]:
tests = [
  ("What is the voluntary lock-in facility for mutual fund folios?", "Voluntary"),
  ("Can mutual funds borrow money?", "Borrowing"),
  ("How should gold and silver held by mutual funds be valued?", "Gold"),
  ("What must social media posts by SEBI-regulated entities disclose?", "Social"),
  ("What is the RBI repo rate decision in October 2026?", "PR1266"),
  ("What did the Financial Stability Report say in June 2026?", "PR5720"),
]
hits = 0
for q, expect in tests:
    r = col.query(query_embeddings=model.encode([q], normalize_embeddings=True).tolist(), n_results=3)
    top = r["metadatas"][0][0]["doc"]
    ok = expect.lower() in top.lower()
    hits += ok
    print("OK " if ok else "MISS", q, "->", top[:70])
print(f"{hits}/{len(tests)} top hits")
json.dump({"documents": docs, "chunks": len(chunks), "top_hits": f"{hits}/{len(tests)}"},
          open("index_stats.json", "w"), indent=2)

OK  What is the voluntary lock-in facility for mutual fund folios? -> sebi_Mar_06_2026_Introduction_of_Voluntary_Lock-in_Debit_freeze_facili
OK  Can mutual funds borrow money? -> sebi_Mar_13_2026_Borrowing_by_Mutual_Funds.pdf
OK  How should gold and silver held by mutual funds be valued? -> sebi_Feb_26_2026_Valuation_of_physical_Gold_and_Silver_held_by_mutual_
MISS What must social media posts by SEBI-regulated entities disclose? -> sebi_Feb_26_2026_Ease_of_Doing_Investment_EoDI_-_Disclosure_of_registe
OK  What is the RBI repo rate decision in October 2026? -> PR1266EE06BBCD7F5A4B2091FB59ADE87FE4A1.pdf
OK  What did the Financial Stability Report say in June 2026? -> PR5720E56F0A4611A404EACBD71F2207673A2.pdf
5/6 top hits
